In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import sys


In [ ]:
print(pl.thread_pool_size()) 

# Read gene-PAS info

In [ ]:
import yaml

base_path = Path('../..').resolve()
with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

tsv_path = base_path / cfg['pas_info_tsv']
df_pas_gene = pd.read_table(tsv_path, index_col=0, dtype={1: 'category'})

pas_to_visualname  = dict(zip(df_pas_gene['PAS_name'], df_pas_gene['visual_name']))
pas_to_repr_genenames = dict(zip(df_pas_gene['PAS_name'], df_pas_gene['gene_name']))
pas_to_region    = dict(zip(df_pas_gene['PAS_name'], df_pas_gene['pas_region']))
visname_to_order   = dict(zip(df_pas_gene['visual_name'], df_pas_gene['PAS_index_from5p']))
visname_to_gene    = dict(zip(df_pas_gene['visual_name'], df_pas_gene['gene_name']))

# Check each table

## Do same for all samples

In [ ]:
PSB_LEVEL = 'class'
psb_paths = list(Path(f'../psb/count_no_cutoff/{PSB_LEVEL}/').glob('*.csv.gz'))
psb_paths

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages('UMI_hist_all.pdf') as pdf:
    for path in psb_paths:
        ct = path.name.removeprefix('PAS_read_count_').removesuffix('_Channel.csv.gz')
        df = pl.read_csv(path).to_pandas().set_index('PAS_id')

        fig, axes = plt.subplots(1, 2, figsize=(8, 3), gridspec_kw={'wspace': 0.3})
        ax1, ax2 = axes
        np.log10(df.sum()+1).hist(bins=20, ax=ax1)
        ax1.set_ylabel('Number of cells')
        ax1.set_xlabel('log10(total UMI cnt)+1')
        ax1.set_title('Per cell barcode')
        np.log10(df.sum(axis=1)+1).hist(bins=20, ax=ax2)
        ax2.set_ylabel('Number of PAS')
        ax2.set_xlabel('log10(total UMI cnt + 1)')
        ax2.set_title('Per PAS')
        fig.suptitle(ct)

        pdf.savefig(fig, bbox_inches='tight')
        fig.savefig(f'UMI_hist_{ct}.png', bbox_inches='tight')
        plt.show()
        plt.close(fig)

# Load PAS assigned read counts

In [ ]:
import ast

In [ ]:
lines = {}
with open('../stats/merged_count.txt', 'r') as cntfile:
    for line in cntfile.readlines():
        fields = line.strip().split('\t')
        sample = fields[0]
        dict_str = fields[1]
        lines[sample] = ast.literal_eval(dict_str.removeprefix('Counter(').removesuffix(')'))

df_assigned_cnt = pd.DataFrame(lines).T
df_assigned_cnt.columns = [c.removeprefix('XS:Z:') for c in df_assigned_cnt.columns]
df_assigned_cnt

In [ ]:
df_assigned_cnt['pct_assigned'] = df_assigned_cnt['Assigned'] / (df_assigned_cnt['Assigned'] + df_assigned_cnt['Unassigned_NoFeatures']) * 100

fig, axes = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'wspace': 0.2})

# 1. Histogram + KDE
df_assigned_cnt['pct_assigned'].plot(kind='hist', bins=30, density=True, alpha=0.5, ax=axes[0])
df_assigned_cnt['pct_assigned'].plot(kind='kde', ax=axes[0], color='black')
axes[0].set_xlabel('% Assigned')
axes[0].set_title('Distribution')

# 2. Sorted rank plot
sorted_df_assigned_cnt = df_assigned_cnt.sort_values('pct_assigned').reset_index()
axes[1].scatter(range(len(sorted_df_assigned_cnt)), sorted_df_assigned_cnt['pct_assigned'], s=8)
axes[1].set_xlabel('Sample rank')
axes[1].set_ylabel('% Assigned')
axes[1].set_title('Per-sample (sorted)')
axes[1].grid(alpha=0.3)

plt.savefig('pct_assigned_summary.png', bbox_inches='tight')
plt.show()